# JPEG cue after Grounding DINO preprocessing
Fixed Step 0 CPU experiment on the 100 sources from the timing prototype. Draw a balanced 60/40 split once. Fit separate standardised logistic regressions on pixel statistics for Q95 vs Q30 (positive control), then Q95 vs Q90 (mild cue). No validation tuning, paired-input features or detector weights. Evaluate single views at a fixed 0.5 threshold. Gate each comparison on lower 95% source-bootstrap accuracy > 0.50 and exact one-sided within-source label-swap p < 0.05. Interpret the mild test only after the positive control passes.

In [ ]:
from time import perf_counter
start = perf_counter()
import csv
import io
import json
import math
import random
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1"], check=True)
import numpy as np
import PIL
from PIL import Image
from scipy.fft import dctn
from scipy.stats import binomtest
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import AutoImageProcessor

source_ids = [[22623,0],[142585,0],[565624,0],[195918,0],[89697,0],[476704,1],[184324,1],[223738,0],[554328,0],[5001,1],[350003,1],[534664,0],[228436,1],[289415,0],[123213,0],[30504,0],[306437,1],[384527,1],[135670,1],[164115,1],[460682,0],[573094,1],[8211,1],[490936,1],[193926,1],[7795,0],[394559,0],[451043,0],[109055,1],[254814,1],[491213,0],[507797,0],[144333,1],[8899,1],[36844,1],[82821,1],[196843,1],[102411,0],[319607,1],[132375,1],[421060,0],[169996,1],[46872,0],[210273,1],[152771,1],[301061,0],[309391,1],[546823,1],[389804,0],[157756,1],[470173,1],[449996,0],[458223,1],[414510,1],[219271,0],[414340,0],[65350,1],[190753,1],[575357,0],[425227,0],[512836,0],[550691,0],[226417,1],[101762,1],[70774,1],[277005,1],[471893,0],[274687,1],[390555,0],[153527,1],[460229,0],[162858,1],[468245,0],[245173,1],[403565,1],[514376,0],[545594,0],[428111,1],[416991,0],[263463,0],[394328,0],[117645,0],[97988,1],[494913,0],[281929,1],[130386,1],[87038,1],[244411,0],[129113,0],[487583,0],[411754,0],[468501,0],[479912,0],[150417,0],[537672,0],[40083,1],[370208,1],[378139,1],[252701,0],[502347,0]]
split_rng = random.Random(5142026)
groups = {}
for positive in (0, 1):
    ids = sorted(i for i, flag in source_ids if flag == positive)
    assert len(ids) == 50
    split_rng.shuffle(ids)
    groups.update({i: ("train" if n < 30 else "held_out", positive) for n, i in enumerate(ids)})
assert len(groups) == 100
out = Path("/kaggle/working")
inputs = Path("/kaggle/input")
root = next(inputs.rglob("instances_val2017.json")).parent.parent / "val2017"
revision = "a2bb814dd30d776dcf7e30523b00659f4f141c71"
processor = AutoImageProcessor.from_pretrained("IDEA-Research/grounding-dino-tiny", revision=revision, use_fast=False)
print(type(processor).__name__, processor.size, flush=True)

In [ ]:
def pixel_features(chw):
    # Use only detector-input pixels. Tile in detector coordinates; ignore partial edge tiles.
    r, g, b = chw
    values = []
    for plane in ((r + 2*g + b)/4, r-g, b-g):
        dx = np.diff(plane, axis=1)
        dy = np.diff(plane, axis=0)
        for grad, axis in ((dx, 1), (dy, 0)):
            curvature = np.abs(np.diff(grad, axis=axis))
            values.extend(np.log1p(np.quantile(curvature, [0.25, 0.5, 0.75, 0.9, 0.99])))
            profile = curvature.mean(axis=1-axis)
            # Periodic changes in edge strength along each axis reflect processing artefacts.
            freq = np.fft.rfftfreq(len(profile))
            power = np.abs(np.fft.rfft((profile-profile.mean()) * np.hanning(len(profile))))**2
            power /= power.sum() + 1e-12
            for lo, hi in ((0.025,0.05),(0.05,0.09),(0.09,0.15),(0.15,0.25),(0.25,0.51)):
                band = power[(freq >= lo) & (freq < hi)]
                values.extend((np.log(band.sum()+1e-12), np.log(band.max(initial=0)+1e-12)))
        h, w = plane.shape
        hh, ww = h//8, w//8
        blocks = plane[:8*hh, :8*ww].reshape(hh,8,ww,8).transpose(0,2,1,3)
        coeff = np.abs(dctn(blocks, axes=(-2,-1), norm="ortho"))
        energy = coeff.mean(axis=(0,1)).reshape(-1)[1:]
        values.extend(np.log((energy + 1e-8)/(energy.mean() + 1e-8)))
    return np.array(values, dtype=np.float64)

qualities = (95, 30, 90)
features = {}
for n, image_id in enumerate(sorted(groups), 1):
    with Image.open(root / f"{image_id:012d}.jpg") as original:
        rgb = original.convert("RGB")
    for q in qualities:
        buffer = io.BytesIO()
        rgb.save(buffer, format="JPEG", quality=q, subsampling=2, optimize=False)
        buffer.seek(0)
        with Image.open(buffer) as encoded:
            pixels = processor(images=encoded.convert("RGB"), return_tensors="np").pixel_values[0]
        features[image_id, q] = pixel_features(pixels)
    if n % 20 == 0:
        print(f"Completed {n}/100 sources", flush=True)
assert len(features) == 300
assert np.isfinite(np.stack(list(features.values()))).all()

In [ ]:
import scipy
import transformers
train = sorted(i for i in groups if groups[i][0] == "train")
held_out = sorted(i for i in groups if groups[i][0] == "held_out")
assert len(train) == 60 and len(held_out) == 40 and set(train).isdisjoint(held_out)
report = {"sources_train": len(train), "sources_held_out": len(held_out), "views": len(features),
          "device": "CPU", "checkpoint_revision": revision, "processor": type(processor).__name__,
          "processor_size": processor.size, "pillow": PIL.__version__, "transformers": transformers.__version__,
          "scikit_learn": sklearn.__version__, "scipy": scipy.__version__, "features": len(next(iter(features.values()))),
          "criteria": "95% source-bootstrap lower bound > 0.5 and exact one-sided within-source swap p < 0.05; gate mild on strong", "comparisons": {}}
predictions = []
for lower_q in (30, 90):
    pair = (95, lower_q)
    def matrix(ids):
        return np.stack([features[i, q] for i in ids for q in pair])
    y_train = np.tile([0, 1], len(train))
    model = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, solver="liblinear", max_iter=2000, random_state=0))
    model.fit(matrix(train), y_train)
    probabilities = model.predict_proba(matrix(held_out))[:, 1].reshape(-1, 2)
    predicted = (probabilities >= 0.5).astype(int)
    correct = predicted == np.array([0, 1])
    n_correct = correct.sum(axis=1)
    both = int((n_correct == 2).sum())
    neither = int((n_correct == 0).sum())
    p = binomtest(both, both+neither, 0.5, alternative="greater").pvalue if both+neither else 1.0
    # Stratified source-cluster bootstrap, keeping the two JPEG variants together.
    rng = np.random.default_rng(511)
    bootstrap = np.zeros(20000)
    for flag in (0, 1):
        scores = n_correct[[groups[i][1] == flag for i in held_out]] / 2
        assert len(scores) == 20
        bootstrap += rng.choice(scores, size=(20000, len(scores)), replace=True).mean(axis=1)/2
    low, high = np.quantile(bootstrap, [0.025, 0.975])
    report["comparisons"][f"95_vs_{lower_q}"] = {
        "correct": int(correct.sum()), "views": int(correct.size), "accuracy": float(correct.mean()),
        "source_bootstrap_95": [float(low), float(high)], "both_correct_sources": both,
        "neither_correct_sources": neither, "swap_p": float(p), "passes": bool(low > 0.5 and p < 0.05),
        "correct_by_quality": {str(q):int(correct[:, j].sum()) for j,q in enumerate(pair)}}
    for n, i in enumerate(held_out):
        for j, q in enumerate(pair):
            predictions.append([i, groups[i][1], lower_q, q, float(probabilities[n,j]), int(correct[n,j])])

with (out / "held_out_predictions.csv").open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["source_id","bicycle_positive","comparison_lower_quality","true_quality","prob_lower_quality","correct"])
    writer.writerows(predictions)
with (out / "step0_split.csv").open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["source_id","split","bicycle_positive"])
    writer.writerows([i, *groups[i]] for i in sorted(groups))
report["runtime_seconds"] = perf_counter() - start
(out / "step0_report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))